In [28]:
import os
import sys
from importlib.metadata import metadata
from pathlib import Path
from skimage import filters

# Automatically locate and add the 'src' folder to sys.path
project_root = Path(os.path.abspath(''))
src_path = project_root / "src"
if src_path not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd

import cupy as cp

from imaging import imsave, imread, imshow, using

In [38]:
catalog_0 = pd.read_csv('/data_local/LabVF/PESA-Brain/Auxiliary/PESABrain_SubjectCatalog_0.csv')
catalog_1 = pd.read_excel('/data_local/LabVF/PESA-Brain/Auxiliary/PESABrain_SubjectCatalog_1.xlsx')
catalog_2 = pd.read_excel('/data_local/LabVF/PESA-Brain/Auxiliary/PESABrain_SubjectCatalog_2_Enfermeria.xlsx')


In [30]:
catalog_0.head()

,MR ID,Date,Subject,Age,Scanner,Scans
0,BMRI100102,2024-10-15,PESA14100025,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), COR(1), DT..."
1,BMRI101198,2022-09-30,PESA5184729,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), CO 3D T1(1..."
2,BMRI101846,2022-02-15,PESA87616,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), DTI NODDI(..."
3,BMRI103891,2024-11-12,PESA7001316,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), AP 4DQflow..."
4,BMRI104117,2023-02-24,PESA10233601,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), 4DQflowNeu..."


In [31]:
catalog_1.head()

,SEQN,Sexo,FechaNacimiento,IdRaza,VISITA,CodigoImagen,CODIGOPRUEBA_RECONOCIMIENTO_MEDICO,PSQDATE_RECONOCIMIENTO_MEDICO,BMXHT,BMXWT,...,IPAQ133,IPAQ141,PSDANSCO,PSDDESCO,PSDSSSCO,PSDSTSCO,SLQ070B,SLQ130,PSQEDUCA,PSQINCOM
0,28,1,1961-08-02 01:00:00,1,4,PESA841,RE629534,2021-11-24 09:28:15.565,168.0,74.0,...,NaN,660.0,10.0,1.0,30.0,7.0,0.0,2.0,6.0,6.0
1,29,2,1969-05-18 00:00:00,1,4,PESA900,RE536721,2022-04-05 08:09:07.063,160.0,58.2,...,180.0,840.0,26.0,22.0,25.0,28.0,0.0,4.0,6.0,2.0
2,38,1,1958-02-25 01:00:00,1,4,PESA1521,RE435268,2021-01-26 10:02:45.101,178.0,111.0,...,105.0,960.0,13.0,0.0,30.0,5.0,0.0,3.0,3.0,3.0
3,58,1,1956-01-26 00:00:00,1,4,PESA3481,RE364374,2021-02-08 09:19:17.144,170.0,74.2,...,120.0,720.0,15.0,22.0,21.0,22.0,1.0,5.0,5.0,4.0
4,59,1,1956-04-23 01:00:00,1,4,PESA3600,RE389335,2020-11-04 07:56:08.491,180.0,94.9,...,450.0,630.0,13.0,1.0,28.0,9.0,0.0,1.0,5.0,3.0


In [32]:
from glob import glob

res_path = '/home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_IgnacioMarcos/LabVF/PESA-Brain/RESULTS/QVTPlus/'
ids2keep = []
for folder in glob(res_path + '/*'):
    name = folder.split('/')[-1]
    if 'PESA' in name: ids2keep.append(name)

print(f'{len(ids2keep)} IDs found')


400 IDs found


In [39]:
catalog_1['Subject'] = catalog_1['CodigoImagen']
catalog_inter = pd.merge(catalog_0, catalog_1, on='Subject', how='inner')
catalog_inter = pd.merge(catalog_inter, catalog_2, on='SEQN', how='inner')

catalog_inter = catalog_inter[catalog_inter['Subject'].isin(ids2keep)]

catalog_inter['FechaNacimiento'] = pd.to_datetime(catalog_inter['FechaNacimiento'])
catalog_inter['FechaNacimiento'] = catalog_inter['FechaNacimiento'].dt.strftime('%Y-%m-%d')

catalog_inter['age_at_mri'] = (pd.to_datetime(catalog_inter['Date']) - pd.to_datetime(catalog_inter['FechaNacimiento'])).dt.days / 365

In [40]:
catalog_inter

,MR ID,Date,Subject,Age,Scanner,Scans,SEQN,Sexo,FechaNacimiento,IdRaza,...,Descansado,Covid19,Fertil,FechaUltimaMenstruacion,TomaMedicacion6Horas,Ayuno6Horas,UltimaInjestaHoras,AnGap,Comentarios,age_at_mri
3,BMRI103891,2024-11-12,PESA7001316,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), AP 4DQflow...",2645,1,1964-09-18,1,...,1.0,0.0,NaN,NaT,False,1.0,18.0,16,NaN,60.191781
4,BMRI103891,2024-11-12,PESA7001316,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), AP 4DQflow...",2645,1,1964-09-18,1,...,1.0,0.0,NaN,NaT,False,1.0,18.0,16,NaN,60.191781
5,BMRI104117,2023-02-24,PESA10233601,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), 4DQflowNeu...",3198,1,1972-08-27,1,...,1.0,1.0,NaN,NaT,False,1.0,114.0,17,COVID AGOSTO 22,50.528767
6,BMRI104385,2022-12-30,PESA1359556,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), 4DQflowNeu...",1165,1,1963-09-24,1,...,1.0,0.0,NaN,NaT,False,1.0,7.0,14,NaN,59.306849
7,BMRI104820,2024-10-25,PESA6416089,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), AP 4DQflow...",2532,2,1972-05-01,1,...,1.0,1.0,0.0,NaT,False,1.0,14.0,17,NaN,52.520548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
978,BMRI994162,2023-12-19,PESA9897316,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), AP 4DQflow...",3145,1,1968-08-25,1,...,1.0,0.0,NaN,NaT,True,1.0,11.0,16,NaN,55.353425
980,BMRI994731,2025-01-27,PESA14807104,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), 4DQflowNeu...",3847,1,1965-06-06,1,...,1.0,0.0,NaN,NaT,False,1.0,7.0,17,NaN,59.684932
981,BMRI994933,2023-01-13,PESA7661824,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), 4DQflowNeu...",2767,1,1963-09-17,1,...,1.0,1.0,NaN,NaT,True,1.0,13.0,12,Covid + JUNIO 2022,59.364384
986,BMRI999786,2023-06-23,PESA6482116,NaN,DIONISIO,"3D T1(1), 3D_FLAIR(1), 3D_T2 HR(1), 4DQflowNeu...",2545,1,1966-04-04,1,...,1.0,0.0,NaN,NaT,False,1.0,7.0,20,NaN,57.257534


In [ ]:
sex_map = {
    1: 'Male',
    2: 'Female'
}

catalog_out = pd.DataFrame(
    {
        'patient_id': catalog_inter['Subject'],
        'mri_id': catalog_inter['MR ID'],
        'med_recon_id': catalog_inter['CODIGOPRUEBA_RECONOCIMIENTO_MEDICO'],
        'seq_num': catalog_inter['SEQN'],
        'visit': catalog_inter['Visita'],

        'sex': catalog_inter['Sexo'].map(sex_map),
        'age_at_mri': np.round(catalog_inter['age_at_mri'], 2),
        'weight': catalog_inter['Peso'],
        'height': catalog_inter['Talla'],

        'BPXSYM': catalog_inter['BPXSYM'],
        'BPXDIM': catalog_inter['BPXDIM'],
        'BPXPLS': catalog_inter['BPXPLS'], 
        'Hematocrit': catalog_inter['Hematocrito'],
        'TAS': catalog_inter['TAS'],
        'TAD': catalog_inter['TAD'],
    }
)

catalog_out.to_excel('/data_local/LabVF/PESA-Brain/Auxiliary/PESABrain_DB_Batch1.xlsx', index=False)

### BMI & SYS|DIAS Delta

In [46]:
catalog_out['bmi'] = catalog_out['weight'] / (catalog_out['height'] / 100)**2 ## [kg/m2]
catalog_out['sys_dias_delta'] = catalog_out['BPXSYM'] - catalog_out['BPXDIM'] ## [mmHg]

catalog_out

,patient_id,mri_id,med_recon_id,seq_num,visit,sex,age_at_mri,weight,height,BPXSYM,BPXDIM,BPXPLS,Hematocrit,TAS,TAD,bmi,sys_dias_delta
3,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,95.0,65.0,47.0,177.0,105.0,29.821142,61.0
4,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,95.0,65.0,47.0,177.0,105.0,29.821142,61.0
5,PESA10233601,BMRI104117,RE851355,3198,4,Male,50.53,81.5,180.0,126.0,79.0,71.0,48.0,142.0,89.0,25.154321,47.0
6,PESA1359556,BMRI104385,RE187620,1165,4,Male,59.31,90.0,174.0,118.0,68.0,55.0,49.0,121.0,77.0,29.726516,50.0
7,PESA6416089,BMRI104820,RE344145,2532,4,Female,52.52,61.0,162.0,105.0,58.0,62.0,39.0,120.0,70.0,23.243408,47.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
978,PESA9897316,BMRI994162,RE248777,3145,4,Male,55.35,85.0,185.0,107.0,68.0,82.0,52.0,130.0,82.0,24.835646,39.0
980,PESA14807104,BMRI994731,RE643483,3847,4,Male,59.68,93.0,182.0,147.0,94.0,64.0,47.0,138.0,84.0,28.076319,53.0
981,PESA7661824,BMRI994933,RE342497,2767,4,Male,59.36,93.0,173.0,138.0,90.0,52.0,49.0,138.0,84.0,31.073541,48.0
986,PESA6482116,BMRI999786,RE874997,2545,4,Male,57.26,88.0,170.0,141.0,86.0,57.0,49.0,144.0,85.0,30.449827,55.0


In [47]:
catalog_out.to_excel('/data_local/LabVF/PESA-Brain/Auxiliary/PESABrain_DB_Batch1.xlsx', index=False)

## ASL DATA

In [45]:
asl_catalog = pd.read_excel('/home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_IgnacioMarcos/LabVF/PESA-Brain/PESABrain_CBF_ASL_Results.xlsx')
asl_catalog

,region,pesa_id,mri_id,raw_mean,thresholded_mean,voxels,cov
0,Left-Cerebral-White-Matter,PESA14100025,BMRI100102,22.686265,22.686265,11986,0.767378
1,Left-Lateral-Ventricle,PESA14100025,BMRI100102,21.668488,21.389909,645,1.278339
2,Left-Inf-Lat-Vent,PESA14100025,BMRI100102,72.771677,72.771677,51,0.773744
3,Left-Cerebellum-White-Matter,PESA14100025,BMRI100102,28.880551,28.880551,667,0.564265
4,Left-Cerebellum-Cortex,PESA14100025,BMRI100102,39.379821,39.379821,1794,0.517728
...,...,...,...,...,...,...,...
226110,ctx-Anterior-Cingulate,PESA13461561,IA997404,75.738216,73.746056,371,0.591886
226111,ctx-Posterior-Cingulate,PESA13461561,IA997404,101.815959,94.287681,288,0.532772
226112,ctx-Left-Hemisphere,PESA13461561,IA997404,69.417361,67.883176,10797,0.610397
226113,ctx-Right-Hemisphere,PESA13461561,IA997404,70.725918,69.040184,11284,0.596231


In [56]:
roi_names = ['ctx-whole-brain', 'ctx-left-hemisphere', 'ctx-right-hemisphere']

# New df where we have the patient_id, the thresholded_mean for each roi (Current df have the rois as rows)
asl_catalog_rois = pd.DataFrame(
    {
        'patient_id': asl_catalog['pesa_id'],
        roi_names[0]: asl_catalog['thresholded_mean'][asl_catalog['region'].str.lower() == roi_names[0].lower()],
        roi_names[1]: asl_catalog['thresholded_mean'][asl_catalog['region'].str.lower() == roi_names[1].lower()],
        roi_names[2]: asl_catalog['thresholded_mean'][asl_catalog['region'].str.lower() == roi_names[2].lower()]
    }
)

asl_catalog_rois

,patient_id,ctx-whole-brain,ctx-left-hemisphere,ctx-right-hemisphere
0,PESA14100025,NaN,NaN,NaN
1,PESA14100025,NaN,NaN,NaN
2,PESA14100025,NaN,NaN,NaN
3,PESA14100025,NaN,NaN,NaN
4,PESA14100025,NaN,NaN,NaN
...,...,...,...,...
226110,PESA13461561,NaN,NaN,NaN
226111,PESA13461561,NaN,NaN,NaN
226112,PESA13461561,NaN,67.883176,NaN
226113,PESA13461561,NaN,NaN,69.040184


In [57]:
# Group by patient to avoid NaN values
asl_catalog_rois = asl_catalog_rois.groupby('patient_id').agg(lambda x: x.dropna().mean())

asl_catalog_rois


,ctx-whole-brain,ctx-left-hemisphere,ctx-right-hemisphere
patient_id,,,
PESA10017225,62.389454,61.897253,62.871607
PESA1002001,55.751993,55.773419,55.731381
PESA100489,73.557550,73.555987,73.559161
PESA10055241,55.091414,55.306647,54.878386
PESA1006009,64.850706,63.393821,66.288385
...,...,...,...
PESA9935104,56.903359,57.807313,55.979328
PESA9947716,72.428656,69.965742,74.834474
PESA9954025,59.616568,59.377463,59.850783


In [ ]:
catalog_out = pd.merge(catalog_out, asl_catalog_rois, on='patient_id', how='inner')
catalog_out.to_excel('/data_local/LabVF/PESA-Brain/Auxiliary/PESABrain_DB_Batch1.xlsx', index=False)

In [60]:
catalog_out

,patient_id,mri_id,med_recon_id,seq_num,visit,sex,age_at_mri,weight,height,BPXSYM,BPXDIM,BPXPLS,Hematocrit,TAS,TAD,bmi,sys_dias_delta,ctx-whole-brain,ctx-left-hemisphere,ctx-right-hemisphere
0,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,95.0,65.0,47.0,177.0,105.0,29.821142,61.0,49.248298,50.282223,48.231163
1,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,95.0,65.0,47.0,177.0,105.0,29.821142,61.0,49.248298,50.282223,48.231163
2,PESA10233601,BMRI104117,RE851355,3198,4,Male,50.53,81.5,180.0,126.0,79.0,71.0,48.0,142.0,89.0,25.154321,47.0,58.186166,63.100405,53.141901
3,PESA1359556,BMRI104385,RE187620,1165,4,Male,59.31,90.0,174.0,118.0,68.0,55.0,49.0,121.0,77.0,29.726516,50.0,49.537020,49.439953,49.631425
4,PESA6416089,BMRI104820,RE344145,2532,4,Female,52.52,61.0,162.0,105.0,58.0,62.0,39.0,120.0,70.0,23.243408,47.0,62.988618,62.992665,62.984628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393,PESA9897316,BMRI994162,RE248777,3145,4,Male,55.35,85.0,185.0,107.0,68.0,82.0,52.0,130.0,82.0,24.835646,39.0,61.322542,61.321816,61.323249
394,PESA14807104,BMRI994731,RE643483,3847,4,Male,59.68,93.0,182.0,147.0,94.0,64.0,47.0,138.0,84.0,28.076319,53.0,57.444646,58.565370,56.289545
395,PESA7661824,BMRI994933,RE342497,2767,4,Male,59.36,93.0,173.0,138.0,90.0,52.0,49.0,138.0,84.0,31.073541,48.0,48.648591,48.456894,48.842579
396,PESA6482116,BMRI999786,RE874997,2545,4,Male,57.26,88.0,170.0,141.0,86.0,57.0,49.0,144.0,85.0,30.449827,55.0,53.799239,54.727071,52.859158


## Plaque on carotides data

In [61]:
plaque_catalog = pd.read_excel('/home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_IgnacioMarcos/LabVF/PESA-Brain/PESABrain_Plaque_Carotides.xlsx')
plaque_catalog

,SEQN,VISITA,SEXO,FECHA_NACIMIENTO,IDRAZA,IDESTADO_BANCO,NUMERO_IMAGEN,CODIGOPRUEBA_RECONOCIMIENTO_MEDICO,PSQDATE_RECONOCIMIENTO_MEDICO,BMXHT,...,PSQEDUCA,PSQINCOM,NUMERO_PRUEBA,BURDEN_TOTAL_RAW_CORREGIDO,VOLUMEN_RAW_COR_CAROTIDA_DCHA,VOLUMEN_RAW_COR_CAROTIDA_IZQDA,VOLUMEN_RAW_COR_SUMA_CAROTIDAS,VOLUMEN_RAW_COR_FEMORAL_DCHA,VOLUMEN_RAW_COR_FEMORAL_IZQDA,VOLUMEN_RAW_COR_SUMA_FEMORALES
0,28,1,H,1961-08-02,1,3.0,PESA841,RM0000212,2010-06-23,170.0,...,6.0,5.0,EC0000745,15.688008,15.688008,0.000000,15.688008,0.0,0.0,0.0
1,28,3,H,1961-08-02,1,3.0,PESA841,RM0085517,2017-04-05,170.0,...,6.0,7.0,EC0085009,19.192320,19.192320,0.000000,19.192320,0.0,0.0,0.0
2,29,1,M,1969-05-18,1,2.0,PESA900,RM0009175,2010-06-23,160.0,...,6.0,3.0,EC0001280,2.998800,0.000000,2.998800,2.998800,0.0,0.0,0.0
3,29,3,M,1969-05-18,1,2.0,PESA900,RM0089296,2017-07-27,160.0,...,6.0,2.0,EC0089590,2.270520,0.000000,2.270520,2.270520,0.0,0.0,0.0
4,38,1,H,1958-02-25,1,3.0,PESA1521,RM0000932,2010-06-28,177.0,...,4.0,4.0,EC0000690,6.777288,0.000000,6.777288,6.777288,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,4232,3,H,1964-08-19,1,1.0,PESA17918289,RM0113708,2019-03-13,172.0,...,5.0,6.0,EC0113126,10.727136,10.727136,0.000000,10.727136,0.0,0.0,0.0
1996,4233,1,M,1967-02-07,1,1.0,PESA17926756,RM0047481,2014-04-22,170.0,...,6.0,4.0,EC0047689,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
1997,4233,3,M,1967-02-07,1,1.0,PESA17926756,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1998,4236,1,H,1964-05-05,1,3.0,PESA17952169,RM0045110,2014-02-27,182.0,...,5.0,7.0,EC0045375,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0


In [71]:
# Tenemos 2 filas por paciente, V1 y V3, queremos por el moneto V3 (Fecha mayor)
plaque_catalog_sub = pd.DataFrame({
    'patient_id': plaque_catalog['NUMERO_IMAGEN'],
    'plaque_left_carotid': plaque_catalog['VOLUMEN_RAW_COR_CAROTIDA_DCHA'][plaque_catalog['VISITA'] == 3],
    'plaque_right_carotid': plaque_catalog['VOLUMEN_RAW_COR_CAROTIDA_IZQDA'][plaque_catalog['VISITA'] == 3],
}).dropna()
plaque_catalog_sub


,patient_id,plaque_left_carotid,plaque_right_carotid
1,PESA841,19.192320,0.000000
3,PESA900,0.000000,2.270520
5,PESA1521,28.908432,0.000000
7,PESA3481,0.000000,0.000000
9,PESA3600,0.000000,16.142112
...,...,...,...
1989,PESA17859076,0.000000,0.000000
1991,PESA17867529,0.000000,48.220704
1993,PESA17875984,0.000000,0.000000
1995,PESA17918289,10.727136,0.000000


In [73]:
catalog_out = pd.merge(catalog_out, plaque_catalog_sub, on='patient_id', how='inner')
catalog_out

,patient_id,mri_id,med_recon_id,seq_num,visit,sex,age_at_mri,weight,height,BPXSYM,...,Hematocrit,TAS,TAD,bmi,sys_dias_delta,ctx-whole-brain,ctx-left-hemisphere,ctx-right-hemisphere,plaque_left_carotid,plaque_right_carotid
0,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,...,47.0,177.0,105.0,29.821142,61.0,49.248298,50.282223,48.231163,0.000000,38.881584
1,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,...,47.0,177.0,105.0,29.821142,61.0,49.248298,50.282223,48.231163,0.000000,38.881584
2,PESA10233601,BMRI104117,RE851355,3198,4,Male,50.53,81.5,180.0,126.0,...,48.0,142.0,89.0,25.154321,47.0,58.186166,63.100405,53.141901,0.000000,0.000000
3,PESA1359556,BMRI104385,RE187620,1165,4,Male,59.31,90.0,174.0,118.0,...,49.0,121.0,77.0,29.726516,50.0,49.537020,49.439953,49.631425,0.000000,0.000000
4,PESA6416089,BMRI104820,RE344145,2532,4,Female,52.52,61.0,162.0,105.0,...,39.0,120.0,70.0,23.243408,47.0,62.988618,62.992665,62.984628,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,PESA9897316,BMRI994162,RE248777,3145,4,Male,55.35,85.0,185.0,107.0,...,52.0,130.0,82.0,24.835646,39.0,61.322542,61.321816,61.323249,32.961096,0.000000
390,PESA14807104,BMRI994731,RE643483,3847,4,Male,59.68,93.0,182.0,147.0,...,47.0,138.0,84.0,28.076319,53.0,57.444646,58.565370,56.289545,0.000000,0.000000
391,PESA7661824,BMRI994933,RE342497,2767,4,Male,59.36,93.0,173.0,138.0,...,49.0,138.0,84.0,31.073541,48.0,48.648591,48.456894,48.842579,0.000000,0.000000
392,PESA6482116,BMRI999786,RE874997,2545,4,Male,57.26,88.0,170.0,141.0,...,49.0,144.0,85.0,30.449827,55.0,53.799239,54.727071,52.859158,16.236360,0.000000


In [74]:
catalog_out.to_excel('/data_local/LabVF/PESA-Brain/Auxiliary/PESABrain_DB_Batch1.xlsx', index=False)
catalog_out

,patient_id,mri_id,med_recon_id,seq_num,visit,sex,age_at_mri,weight,height,BPXSYM,...,Hematocrit,TAS,TAD,bmi,sys_dias_delta,ctx-whole-brain,ctx-left-hemisphere,ctx-right-hemisphere,plaque_left_carotid,plaque_right_carotid
0,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,...,47.0,177.0,105.0,29.821142,61.0,49.248298,50.282223,48.231163,0.000000,38.881584
1,PESA7001316,BMRI103891,RE723265,2645,4,Male,60.19,87.2,171.0,156.0,...,47.0,177.0,105.0,29.821142,61.0,49.248298,50.282223,48.231163,0.000000,38.881584
2,PESA10233601,BMRI104117,RE851355,3198,4,Male,50.53,81.5,180.0,126.0,...,48.0,142.0,89.0,25.154321,47.0,58.186166,63.100405,53.141901,0.000000,0.000000
3,PESA1359556,BMRI104385,RE187620,1165,4,Male,59.31,90.0,174.0,118.0,...,49.0,121.0,77.0,29.726516,50.0,49.537020,49.439953,49.631425,0.000000,0.000000
4,PESA6416089,BMRI104820,RE344145,2532,4,Female,52.52,61.0,162.0,105.0,...,39.0,120.0,70.0,23.243408,47.0,62.988618,62.992665,62.984628,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,PESA9897316,BMRI994162,RE248777,3145,4,Male,55.35,85.0,185.0,107.0,...,52.0,130.0,82.0,24.835646,39.0,61.322542,61.321816,61.323249,32.961096,0.000000
390,PESA14807104,BMRI994731,RE643483,3847,4,Male,59.68,93.0,182.0,147.0,...,47.0,138.0,84.0,28.076319,53.0,57.444646,58.565370,56.289545,0.000000,0.000000
391,PESA7661824,BMRI994933,RE342497,2767,4,Male,59.36,93.0,173.0,138.0,...,49.0,138.0,84.0,31.073541,48.0,48.648591,48.456894,48.842579,0.000000,0.000000
392,PESA6482116,BMRI999786,RE874997,2545,4,Male,57.26,88.0,170.0,141.0,...,49.0,144.0,85.0,30.449827,55.0,53.799239,54.727071,52.859158,16.236360,0.000000
